In [1]:
import sys
sys.path.append('/home/jovyan/work/tactics_demo/tools')
import os
import glob

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [4]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [5]:
instrument_id = '511520'
trade_ymd = '20260319'

In [6]:
param_dict = {

    'name' : 'TBM',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,
    
    'x_window' : 300 ,
    'y_window' : 300 ,
    'stride': 60,

    'k_up': 0.5,
    'k_down': 0.5
}

In [7]:
snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
print(snap_list[1])

{'time_mark': 1773883801000, 'time_open': 1773883800010, 'time_last': 1773883800430, 'price_open': 116.01, 'price_low': 116.003, 'price_high': 116.01, 'price_last': 116.003, 'price_vwap': 116.0097, 'num_trades': 11, 'num_buy': 0, 'num_sell': 2, 'buy_trade': [], 'sell_trade': [[116.003, 500], [116.01, 600]], 'bid_insert': [[116.007, 5100], [115.909, 6000], [115.908, 6000]], 'ask_insert': [[116.017, 5100], [116.05, 2656], [116.115, 6000], [116.116, 6000]], 'bid_cancel': [[115.852, 11900]], 'ask_cancel': [[116.02, 5500], [116.115, 6000], [116.116, 6000], [116.2, 11900]], 'bid_book': [[116.007, 5100], [116.003, 51500], [116.001, 20000], [115.996, 100], [115.965, 30000]], 'ask_book': [[116.015, 30000], [116.017, 5100], [116.02, 5400], [116.025, 400], [116.026, 400]], 'history': False}


In [ ]:
import numpy as np
import pandas as pd

class TrainValidTest():
    def __init__(self, snap_list, param_dict,feature_func,y_func):
        if param_dict is not None:
            self.__dict__.update(param_dict)
        # 确保必要属性存在
        if not hasattr(self, 'x_window'):
            self.x_window = 1
        if not hasattr(self, 'y_window'):
            self.y_window = 1

        self.snap_list = snap_list.copy()
        self.create_feature = feature_func
        self.create_y = y_func

    def samples(self):
        X_list, y_list = [], []
        n = len(self.snap_list)
        stride = getattr(self, 'stride', 1)

        for i in range(self.x_window, n - self.y_window, stride):
            x = self.create_feature(self.snap_list[i - self.x_window:i])
            volatility = x.get('volatility', 0.0)
            y = self.create_y(self.snap_list[i:i + self.y_window], volatility)
            X_list.append(x)
            y_list.append(y)

        X_all = pd.concat(X_list, axis=0, ignore_index=True)
        y_all = pd.concat(y_list, axis=0, ignore_index=True)

        return X_all, y_all

def samples_from_dates(dates, instrument_id, param_dict, create_feature, create_y):
    X_all_list = []
    y_all_list = []
    
    for date in dates:
        try:
            snap_list = base_tool.snap_list_load(instrument_id, date)
            if len(snap_list) < param_dict['x_window'] + param_dict['y_window']:
                print(f"{date}: 数据不足，跳过")
                continue
            tv = TrainValidTest(snap_list, param_dict, create_feature, create_y)
            X_day, y_day = tv.samples()
            X_all_list.append(X_day)
            y_all_list.append(y_day)
            print(f"{date}: 产生 {len(X_day)} 个样本")
        except Exception as e:
            print(f"{date}: 加载失败 - {e}")
    
    if X_all_list:
        X_total = pd.concat(X_all_list, axis=0, ignore_index=True)
        y_total = pd.concat(y_all_list, axis=0, ignore_index=True)
    else:
        X_total = pd.DataFrame()
        y_total = pd.Series()
    
    return X_total, y_total

def create_feature(snap_slice):
    """
    从特征窗口快照切片中提取特征，并计算波动率。
    """
    last = snap_slice[-1]

    price = last['price_last']
    trades = last['num_trades']
    best_bid = last['bid_book'][0][0] if last['bid_book'] else np.nan
    best_ask = last['ask_book'][0][0] if last['ask_book'] else np.nan
    spread = best_ask - best_bid if not np.isnan(best_ask) and not np.isnan(best_bid) else np.nan
    bid_depth = last['bid_book'][0][1] if last['bid_book'] else 0
    ask_depth = last['ask_book'][0][1] if last['ask_book'] else 0

    # 基于特征窗口内的价格序列计算波动率（相对波动率 = 标准差 / 均值）
    prices = [snap['price_last'] for snap in snap_slice if snap['price_last'] is not None]
    if len(prices) >= 2:
        mean_price = np.mean(prices)
        std_price = np.std(prices)
        volatility = std_price / mean_price if mean_price != 0 else 0.0
    else:
        volatility = 0.0

    features = {
        'price_last': price,
        'num_trades': trades,
        'best_bid': best_bid,
        'best_ask': best_ask,
        'spread': spread,
        'bid_depth1': bid_depth,
        'ask_depth1': ask_depth,
        'volatility': volatility,   # 波动率作为特征之一
    }
    return pd.Series(features)

def create_y(snap_slice, volatility):
    """
    根据标签窗口快照切片和波动率（来自特征窗口）计算标签。
    """
    up_factor = 1.25
    down_factor = 0.8

    first_price = snap_slice[0]['price_last']
    if first_price is None:
        return pd.Series([0], index=['label'])

    # 上下轨基于特征窗口波动率计算
    up = first_price * (1 + volatility * up_factor)
    down = first_price * (1 - volatility * down_factor)

    t_up = len(snap_slice)
    t_down = len(snap_slice)

    for i, now in enumerate(snap_slice):
        price = now['price_last']
        if price is None:
            continue
        if price > up and t_up == len(snap_slice):
            t_up = i
        if price < down and t_down == len(snap_slice):
            t_down = i

    if t_up < t_down:
        return pd.Series([1], index=['label'])
    elif t_up > t_down:
        return pd.Series([-1], index=['label'])
    else:
        return pd.Series([0], index=['label'])

## XGBoost 模型训练

In [11]:
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report

instrument_id = '511520'
train_days = 20
valid_days = 4
test_days = 5



trade_dates = ['20260202', '20260203', '20260204', '20260205', '20260206',
                '20260209', '20260210', '20260211', '20260212', '20260213',
                '20260224', '20260225', '20260226', '20260227', '20260302',
                '20260303', '20260304', '20260305', '20260306', '20260309',
                '20260310', '20260311', '20260312', '20260313', '20260316',
                '20260317', '20260318', '20260319', '20260320']
print(f"总交易日数量: {len(trade_dates)}")
print(f"交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}")

总交易日数量: 29
交易日范围: 20260202 ~ 20260320


In [12]:
train_dates = trade_dates[:train_days]
valid_dates = trade_dates[train_days:train_days + valid_days]
test_dates = trade_dates[train_days + valid_days:train_days + valid_days + test_days]

print(f"训练集: {train_dates[0]} ~ {train_dates[-1]} ({len(train_dates)}天)")
print(f"验证集: {valid_dates[0]} ~ {valid_dates[-1]} ({len(valid_dates)}天)")
print(f"测试集: {test_dates[0]} ~ {test_dates[-1]} ({len(test_dates)}天)")

训练集: 20260202 ~ 20260309 (20天)
验证集: 20260310 ~ 20260313 (4天)
测试集: 20260316 ~ 20260320 (5天)


In [13]:
%%time
param_dict = {
    'name': 'TBM',
    'instrument_id': instrument_id,
    'x_window': 300,
    'y_window': 300,
    'stride': 60,
    'k_up': 0.5,
    'k_down': 0.5
}

print("生成训练集样本...")
X_train, y_train = samples_from_dates(train_dates, instrument_id, param_dict, create_feature, create_y)
print(f"训练集样本: X={X_train.shape}, y={y_train.shape}")
if len(y_train) > 0:
    print(f"标签分布:\n{y_train.value_counts()}")

生成训练集样本...
20260202: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260203: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260204: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260205: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260206: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260209: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260210: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260211: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260212: 加载失败 - cannot concatenate object of type '<class 'int'>'; only Series and DataFrame objs are valid
20260213

In [ ]:
%%time
print("生成验证集样本...")
X_valid, y_valid = samples_from_dates(valid_dates, instrument_id, param_dict, create_feature, create_y)
print(f"验证集样本: X={X_valid.shape}, y={y_valid.shape}")
if len(y_valid) > 0:
    print(f"标签分布:\n{y_valid.value_counts()}")

In [ ]:
%%time
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=3,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=10
)

In [ ]:
y_pred = model.predict(X_valid)
print("验证集准确率:", accuracy_score(y_valid, y_pred))
print("\n分类报告:")
print(classification_report(y_valid, y_pred, target_names=['下跌(-1)', '持平(0)', '上涨(1)']))

In [ ]:
import matplotlib.pyplot as plt
xgb.plot_importance(model, max_num_features=10)
plt.title('特征重要性')
plt.tight_layout()
plt.show()

In [ ]:
model_path = f'/home/jovyan/work/backtest_result/{instrument_id}_xgb_model.pkl'
joblib.dump(model, model_path)
print(f"模型已保存至: {model_path}")

## 测试集预测

In [ ]:
%%time
print("生成测试集样本...")
X_test, y_test = samples_from_dates(test_dates, instrument_id, param_dict, create_feature, create_y)
print(f"测试集样本: X={X_test.shape}, y={y_test.shape}")

In [ ]:
y_test_pred = model.predict(X_test)
print("测试集准确率:", accuracy_score(y_test, y_test_pred))
print("\n分类报告:")
print(classification_report(y_test, y_test_pred, target_names=['下跌(-1)', '持平(0)', '上涨(1)']))

## 基于模型的回测

In [ ]:
class StrategyWithModel():
    def __init__(self, model, param_dict={}) -> None:
        self.__dict__.update(param_dict)
        self.model = model
        self.feature_buffer = []
        self.position_last = 0
        self.prev_signal = 0
        
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}_xgb.pkl'
        if os.path.exists(data_file):
            os.remove(data_file)

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']
        
        if price == 0.0 or price == None:
            return
        
        self.feature_buffer.append(snap)
        
        if len(self.feature_buffer) < self.x_window:
            self.position_last = 0
            self.prev_signal = 0
            return
        
        if len(self.feature_buffer) > self.x_window:
            self.feature_buffer.pop(0)
        
        features = create_feature(self.feature_buffer[-self.x_window:])
        X_pred = features.values.reshape(1, -1)
        
        try:
            prediction = self.model.predict(X_pred)[0]
            if prediction != self.prev_signal:
                self.position_last = int(prediction)
                self.prev_signal = int(prediction)
        except Exception as e:
            pass
        return

In [ ]:
def backtest_with_model(instrument_id, trade_ymd, model, strategy_name):
    param_dict = {
        'name': strategy_name,
        'instrument_id': instrument_id,
        'trade_ymd': trade_ymd,
        'x_window': 300,
    }
    strategy = StrategyWithModel(model, param_dict)
    snap_list = base_tool.snap_list_load(instrument_id, trade_ymd)
    position_dict = {}
    for snap in snap_list[:]:
        strategy.on_snap(snap)
        position_dict[snap['time_mark']] = strategy.position_last
    profit = base_tool.backtest_quick(instrument_id, trade_ymd, strategy_name, position_dict)
    return profit

In [ ]:
%%time
test_date = '20260319'
profit = backtest_with_model(instrument_id, test_date, model, 'TBM_XGB')
print(f"日期 {test_date} 回测完成，当日盈亏: {profit}")

In [ ]:
%%time
total_profit = 0
for test_date in test_dates:
    try:
        profit = backtest_with_model(instrument_id, test_date, model, 'TBM_XGB')
        print(f"日期 {test_date} 回测完成，当日盈亏: {profit}")
        total_profit += profit
    except Exception as e:
        print(f"日期 {test_date} 处理失败: {e}")
print(f"\n测试集总盈亏: {total_profit}")

In [ ]:
%%time
from collections import deque
import logging
import os

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        self.__dict__.update(param_dict)
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl' 
        if os.path.exists(data_file):
            os.remove(data_file)

        self.position_last = 0
        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0


        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return

        self.price_list.append(price)

        if(len(self.price_list) < self.long_window):
            self.position_last = 0
            self.prev_signal = 0
            return

        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        long_ma = sum(self.price_list) / self.long_window
        diff = short_ma - long_ma

        if diff > self.threshold:
            current_signal = 1
        elif diff < -self.threshold:
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal:
            self.position_last = current_signal
            self.prev_signal = current_signal

        return


